# 02_ingest_and_profile: load the raw table and profile it
Input: the MySQL table `dataset_list` (connection from src/db.py, see 01_connection_test; the password is entered in a hidden prompt). Set `USE_DATABASE = False` to rerun the notebook from the saved snapshot without connecting. Output: `data/raw/itsm_raw.csv` (git-ignored). Aggregates only: no ticket rows are printed.

## 1. Connect and inspect the schema, keys and constraints
The database's own metadata (INFORMATION_SCHEMA): which tables exist, how each column is defined, and which primary-key, unique, foreign-key constraints and indexes are enforced.

In [1]:
import os, sys
import pandas as pd
from sqlalchemy import text
sys.path.append("../src")
from db import get_engine

USE_DATABASE = True     # False: skip the database and reuse data/raw/itsm_raw.csv from an earlier run
engine = get_engine() if USE_DATABASE else None   # asks for the password in a hidden prompt

Database password:  ········


<details>
<summary><b>Code notes, line by line</b></summary>

- `import os, sys / import pandas as pd / from sqlalchemy import text` : Libraries: file paths, the import path, data frames, and text() for raw SQL.
- `sys.path.append("../src"); from db import get_engine` : Imports the connection helper written by 01_connection_test.
- `USE_DATABASE = True` : Switch: False skips the database and reloads the saved CSV snapshot (for reruns without a connection).
- `engine = get_engine() if USE_DATABASE else None` : connects only when USE_DATABASE is True; the password is asked for in a hidden prompt and is never stored.

</details>

In [2]:
if engine is not None:
    with engine.connect() as conn:
        schema = conn.execute(text("SELECT DATABASE()")).scalar()
        q = lambda sql: pd.read_sql(text(sql), conn, params={"s": schema})
        tables = q("""SELECT TABLE_NAME AS `Table`, TABLE_TYPE AS `Type`
                      FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = :s ORDER BY TABLE_NAME""")
        columns = q("""SELECT ORDINAL_POSITION AS `Position`, COLUMN_NAME AS `Column`, COLUMN_TYPE AS `SQL type`,
                              IS_NULLABLE AS `Nullable`, COLUMN_KEY AS `Key`, COLUMN_DEFAULT AS `Default`
                       FROM INFORMATION_SCHEMA.COLUMNS
                       WHERE TABLE_SCHEMA = :s AND TABLE_NAME = 'dataset_list' ORDER BY ORDINAL_POSITION""")
        constraints = q("""SELECT tc.TABLE_NAME AS `Table`, tc.CONSTRAINT_NAME AS `Constraint`,
                                  tc.CONSTRAINT_TYPE AS `Type`, kcu.COLUMN_NAME AS `Column`,
                                  kcu.REFERENCED_TABLE_NAME AS `Referenced table`,
                                  kcu.REFERENCED_COLUMN_NAME AS `Referenced column`
                           FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS tc
                           LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE kcu
                             ON tc.CONSTRAINT_SCHEMA = kcu.CONSTRAINT_SCHEMA AND tc.TABLE_NAME = kcu.TABLE_NAME
                            AND tc.CONSTRAINT_NAME = kcu.CONSTRAINT_NAME
                           WHERE tc.TABLE_SCHEMA = :s
                           ORDER BY tc.TABLE_NAME, tc.CONSTRAINT_NAME, kcu.ORDINAL_POSITION""")
        indexes = q("""SELECT INDEX_NAME AS `Index`, COLUMN_NAME AS `Column`, NON_UNIQUE AS `Non-unique`
                       FROM INFORMATION_SCHEMA.STATISTICS
                       WHERE TABLE_SCHEMA = :s AND TABLE_NAME = 'dataset_list' ORDER BY INDEX_NAME, SEQ_IN_INDEX""")
        n_rows = conn.execute(text("SELECT COUNT(*) FROM dataset_list")).scalar()

    print("Table 1a. Tables in the database")
    display(tables)
    print("Table 1b. Column definitions of dataset_list")
    display(columns.set_index("Position"))
    print("Table 1c. Primary-key, unique and foreign-key constraints")
    display(constraints if len(constraints) else "None defined")
    print("Table 1d. Indexes on dataset_list")
    display(indexes if len(indexes) else "None defined")
    print("Rows in dataset_list:", n_rows)
else:
    print("Database step skipped (USE_DATABASE = False).")

Table 1a. Tables in the database


,Table,Type
0,dataset_list,BASE TABLE


Table 1b. Column definitions of dataset_list


,Column,SQL type,Nullable,Key,Default
Position,,,,,
1,CI_Name,varchar(255),YES,,None
2,CI_Cat,varchar(255),YES,,None
3,CI_Subcat,varchar(255),YES,,None
4,WBS,varchar(255),YES,,None
5,Incident_ID,varchar(255),YES,,None
6,Status,varchar(255),YES,,None
7,Impact,varchar(255),YES,,None
8,Urgency,varchar(255),YES,,None
9,Priority,varchar(255),YES,,None


Table 1c. Primary-key, unique and foreign-key constraints


'None defined'

Table 1d. Indexes on dataset_list


'None defined'

Rows in dataset_list: 46606


<details>
<summary><b>Code notes, line by line</b></summary>

- `if engine is not None:` : Runs the metadata queries only when connected.
- `with engine.connect() as conn:` : Opens one connection and closes it automatically at the end of the block.
- `schema = conn.execute(text("SELECT DATABASE()")).scalar()` : Asks MySQL for the name of the current database, so the queries need no hard-coded name.
- `q = lambda sql: pd.read_sql(text(sql), conn, params={"s": schema})` : Small helper: runs a query with :s bound to the schema name (a bound parameter, not string pasting) and returns a data frame.
- `tables = q(... INFORMATION_SCHEMA.TABLES ...)` : Lists every table and view in the database (Table 1a).
- `columns = q(... INFORMATION_SCHEMA.COLUMNS ...)` : Column definitions of dataset_list: position, name, SQL type, nullable, key flag and default (Table 1b).
- `constraints = q(... TABLE_CONSTRAINTS LEFT JOIN KEY_COLUMN_USAGE ...)` : Primary-key, unique and foreign-key constraints and the columns they cover; the LEFT JOIN keeps constraints even without column details (Table 1c).
- `indexes = q(... INFORMATION_SCHEMA.STATISTICS ...)` : Indexes on dataset_list and whether each is unique (Table 1d).
- `n_rows = conn.execute(text("SELECT COUNT(*) ...")).scalar()` : Row count, computed in the database.
- `print(...); display(...)` : Shows each table with a numbered title.
- `display(constraints if len(constraints) else "None defined")` : Prints "None defined" instead of an empty table when there are no constraints.
- `else: print("Database step skipped ...")` : Message for offline mode.

</details>

**Result.** The database holds one table, dataset_list, with 46,606 rows. All 25 columns are VARCHAR(255) and nullable, with no default values, and no primary-key, unique or foreign-key constraint is defined (Tables 1b and 1c). The database therefore enforces nothing: numbers and dates are stored as text, blanks are allowed in every column, and Incident_ID is not guaranteed to be unique. These checks are done in the notebooks instead: blanks and placeholders in section 4, Incident_ID uniqueness in section 6, and type conversion in `03_cleaning`.

## 2. Load the table and save the raw snapshot
`data/raw/` is git-ignored.

In [3]:
if engine is not None:
    with engine.connect() as conn:
        df = pd.read_sql(text("SELECT * FROM dataset_list"), conn)
    engine.dispose()                 # close the connection
    os.makedirs("../data/raw", exist_ok=True)
    df.to_csv("../data/raw/itsm_raw.csv", index=False)
else:
    df = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False)
print(df.shape)

(46606, 25)


<details>
<summary><b>Code notes, line by line</b></summary>

- `df = pd.read_sql(text("SELECT * FROM dataset_list"), conn)` : Loads the whole table into a data frame (46,606 rows fit easily in memory).
- `engine.dispose()` : Closes the connection: the database is not needed after this.
- `os.makedirs("../data/raw", exist_ok=True)` : Creates data/raw if it does not exist (git-ignored).
- `df.to_csv("../data/raw/itsm_raw.csv", index=False)` : Saves a raw snapshot, so later notebooks never query the database.
- `else: df = pd.read_csv(..., dtype=str, keep_default_na=False)` : Offline mode: reloads the snapshot with every value as text and blanks kept as empty strings, exactly as stored in MySQL.
- `print(df.shape)` : Rows and columns, to check the load.

</details>

## 3. Column types and visible nulls

In [4]:
print("Shape of dataset:", df.shape)
print("Column names:")
print(df.columns)

print("Table 3a. Data type and non-null count of each column")
df.info()

Shape of dataset: (46606, 25)
Column names:
Index(['CI_Name', 'CI_Cat', 'CI_Subcat', 'WBS', 'Incident_ID', 'Status',
       'Impact', 'Urgency', 'Priority', 'number_cnt', 'Category', 'KB_number',
       'Alert_Status', 'No_of_Reassignments', 'Open_Time', 'Reopen_Time',
       'Resolved_Time', 'Close_Time', 'Handle_Time_hrs', 'Closure_Code',
       'No_of_Related_Interactions', 'Related_Interaction',
       'No_of_Related_Incidents', 'No_of_Related_Changes', 'Related_Change'],
      dtype='str')
Table 3a. Data type and non-null count of each column
<class 'pandas.DataFrame'>
RangeIndex: 46606 entries, 0 to 46605
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   CI_Name                     46606 non-null  str  
 1   CI_Cat                      46606 non-null  str  
 2   CI_Subcat                   46606 non-null  str  
 3   WBS                         46606 non-null  str  
 4   Incident_ID  

<details>
<summary><b>Code notes, line by line</b></summary>

- `print("Shape of dataset:", df.shape)` : Number of rows and columns.
- `print(df.columns)` : The 25 column names, in database order.
- `df.info()` : Non-null count and data type of each column; here all are text, because MySQL stores every column as VARCHAR.

</details>

In [5]:
print("Table 3b. True nulls per column (isna)")
df.isna().sum().sort_values(ascending=False)

Table 3b. True nulls per column (isna)


CI_Name                       0
CI_Cat                        0
CI_Subcat                     0
WBS                           0
Incident_ID                   0
Status                        0
Impact                        0
Urgency                       0
Priority                      0
number_cnt                    0
Category                      0
KB_number                     0
Alert_Status                  0
No_of_Reassignments           0
Open_Time                     0
Reopen_Time                   0
Resolved_Time                 0
Close_Time                    0
Handle_Time_hrs               0
Closure_Code                  0
No_of_Related_Interactions    0
Related_Interaction           0
No_of_Related_Incidents       0
No_of_Related_Changes         0
Related_Change                0
dtype: int64

<details>
<summary><b>Code notes, line by line</b></summary>

- `df.isna().sum()` : Counts true nulls per column.
- `.sort_values(ascending=False)` : Largest counts first; all zero here, because blanks are stored as empty strings or placeholders, not as nulls.

</details>

**Result.** All 25 columns are stored as text (Table 3a: type `str`, 46,606 non-null values in every column), and `isna()` finds no nulls (Table 3b: 0 in every column). Missing values are stored as blanks or placeholders instead (Table 4). Impact, Urgency, Priority, the count columns, Handle_Time_hrs and the four dates are converted in `03_cleaning`.

## 4. Hidden missing values: blanks and placeholders

In [6]:
placeholders = ["", "?", "NA", "N/A", "NaN", "nan", "null", "NULL", "None", "--", "-", "NS", "Unknown"]
hidden = {c: int(df[c].str.strip().isin(placeholders).sum()) for c in df.columns}
print("Table 4. Blank or placeholder values per column")
pd.Series(hidden).sort_values(ascending=False)

Table 4. Blank or placeholder values per column


Related_Change                46046
No_of_Related_Changes         46046
No_of_Related_Incidents       45384
Reopen_Time                   44322
Closure_Code                   2050
Resolved_Time                  1780
Priority                       1380
Impact                         1380
No_of_Related_Interactions      114
CI_Cat                          111
CI_Subcat                       111
Handle_Time_hrs                   1
No_of_Reassignments               1
CI_Name                           0
Status                            0
WBS                               0
Incident_ID                       0
Urgency                           0
Open_Time                         0
KB_number                         0
Alert_Status                      0
number_cnt                        0
Category                          0
Close_Time                        0
Related_Interaction               0
dtype: int64

<details>
<summary><b>Code notes, line by line</b></summary>

- `placeholders = ["", "?", "NA", ...]` : Strings that commonly mean 'missing' in exports.
- `df[c].str.strip().isin(placeholders).sum()` : For each column: removes surrounding spaces, then counts values that are one of the placeholders.
- `hidden = {c: ... for c in df.columns}` : Stores the count per column in a dictionary.
- `pd.Series(hidden).sort_values(ascending=False)` : Shows the hidden missing values, largest first.

</details>

**Result.** Table 4 finds hidden missing values in 13 columns: Related_Change and No_of_Related_Changes are blank on 46,046 tickets (98.8%), No_of_Related_Incidents on 45,384 and Reopen_Time on 44,322. Closure_Code has 2,050 blanks or placeholders (including "Unknown"), Resolved_Time 1,780, Priority and Impact 1,380 each, No_of_Related_Interactions 114, and CI_Cat and CI_Subcat 111 each.

## 5. Cardinality: unique values per column
How many distinct values each column has, how many are blank or a placeholder, and how dominant the most common value is (values themselves are not shown).

In [7]:
card = pd.DataFrame({
    "Unique values": df.nunique(),
    "Unique %": (df.nunique() / len(df) * 100).round(2),
    "Blank or placeholder": pd.Series(hidden),
    "Most common value, share %": df.apply(lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 1)),
}).sort_values("Unique values", ascending=False)
print("Table 5. Cardinality of the raw columns")
display(card)

Table 5. Cardinality of the raw columns


,Unique values,Unique %,Blank or placeholder,"Most common value, share %"
Incident_ID,46606,100.00,0,0.0
number_cnt,46606,100.00,0,0.0
Related_Interaction,43060,92.39,0,7.4
Open_Time,34636,74.32,0,0.0
Close_Time,34528,74.08,0,0.0
Resolved_Time,33628,72.15,1780,3.8
Handle_Time_hrs,30639,65.74,1,0.5
CI_Name,3019,6.48,0,6.5
Reopen_Time,2245,4.82,44322,95.1
KB_number,1825,3.92,0,2.4


<details>
<summary><b>Code notes, line by line</b></summary>

- `"Unique values": df.nunique()` : Number of distinct values per column.
- `"Unique %": ...` : Distinct values as a share of rows: 100% means every row differs (an identifier).
- `"Blank or placeholder": pd.Series(hidden)` : Reuses the counts from section 4.
- `"Most common value, share %": df.apply(lambda s: ...)` : Share of rows holding the most common value: near 100% means the column carries almost no information. Values themselves are not shown.
- `.sort_values("Unique values", ascending=False)` : Highest-cardinality columns first.
- `display(card)` : Table 5.

</details>

**Result.** Incident_ID and number_cnt are unique on every row (Table 5: 46,606 unique values, 100%): an identifier and a noise column. CI_Name (3,019 values), KB_number (1,825) and WBS (274) are high-cardinality codes, and CI_Subcat has 65 values, the most common covering 40.4% of rows. Status (Closed on 100.0% after rounding) and Alert_Status (one value) carry almost no information (last column of Table 5). Decisions: identifiers are excluded from features; categories are one-hot encoded with values seen fewer than 30 times in training pooled as "infrequent"; Alert_Status and number_cnt are dropped (`03_cleaning`); Status is post-event.

## 6. Integrity: unique Incident_ID and duplicate rows

In [8]:
print("Table 6. Incident_ID integrity")
ids = df["Incident_ID"].str.strip()
print("Rows:", len(df), "| unique Incident_ID:", ids.nunique(), "| blank Incident_ID:", int(ids.eq("").sum()))
print("Duplicate Incident_ID rows:", int(ids.duplicated().sum()), "| fully duplicated rows:", int(df.duplicated().sum()))

Table 6. Incident_ID integrity
Rows: 46606 | unique Incident_ID: 46606 | blank Incident_ID: 0
Duplicate Incident_ID rows: 0 | fully duplicated rows: 0


<details>
<summary><b>Code notes, line by line</b></summary>

- `ids = df["Incident_ID"].str.strip()` : Incident IDs without surrounding spaces.
- `ids.nunique() / ids.eq("").sum()` : Distinct IDs and blank IDs.
- `ids.duplicated().sum()` : Rows whose ID already appeared earlier.
- `df.duplicated().sum()` : Rows identical to an earlier row in every column.

</details>

**Result.** 46,606 unique Incident_IDs on 46,606 rows, none blank, and no duplicated IDs or fully duplicated rows (Table 6).

## 7. number_cnt: what is it?

In [9]:
print("Table 7. number_cnt: unique values and summary")
print("Unique number_cnt values:", df["number_cnt"].nunique(), "of", len(df))
print(df["number_cnt"].astype(float).describe().round(3))

Table 7. number_cnt: unique values and summary
Unique number_cnt values: 46606 of 46606
count    46606.000
mean         0.500
std          0.289
min          0.000
25%          0.248
50%          0.500
75%          0.749
max          1.000
Name: number_cnt, dtype: float64


<details>
<summary><b>Code notes, line by line</b></summary>

- `df["number_cnt"].nunique()` : Distinct values: equal to the row count means every value is unique.
- `df["number_cnt"].astype(float).describe().round(3)` : Summary statistics; a mean of 0.5 and quartiles near 0.25 and 0.75 between 0 and 1 are the signature of uniform random numbers.

</details>

**Result.** All 46,606 values are unique and spread evenly between 0 and 1: mean 0.500, quartiles 0.248, 0.500 and 0.749 (Table 7), the pattern of uniform random numbers, not a measurement.

## 8. Small categorical columns and the priority matrix

In [10]:
print("Table 8a. Value counts of the small categorical columns\n")
for c in ["Status", "Impact", "Urgency", "Priority", "Category", "Alert_Status"]:
    print(df[c].value_counts().head(10), "\n")

Table 8a. Value counts of the small categorical columns

Status
Closed              46597
Work in progress        9
Name: count, dtype: int64 

Impact
4     22556
5     16741
3      5234
NS     1380
2       692
1         3
Name: count, dtype: int64 

Urgency
4               22588
5               16779
3                6536
2                 696
1                   6
5 - Very Low        1
Name: count, dtype: int64 

Priority
4     22717
5     16486
3      5323
NA     1380
2       697
1         3
Name: count, dtype: int64 

Category
incident                   37748
request for information     8846
complaint                     11
request for change             1
Name: count, dtype: int64 

Alert_Status
closed    46606
Name: count, dtype: int64 



<details>
<summary><b>Code notes, line by line</b></summary>

- `for c in ["Status", "Impact", ...]:` : Loops over the small categorical columns.
- `df[c].value_counts().head(10)` : Counts of each value (top 10), to spot placeholders such as NS and NA and odd labels such as '5 - Very Low'.

</details>

In [11]:
pm = pd.crosstab([df["Impact"], df["Urgency"]], df["Priority"])
print("Table 8b. Tickets per Impact x Urgency pair (rows) and Priority (columns)")
display(pm)
print("Impact x Urgency pairs that map to more than one priority:", int(((pm > 0).sum(axis=1) > 1).sum()), "of", len(pm))

na, ns = df["Priority"].str.strip().eq("NA"), df["Impact"].str.strip().eq("NS")
print("Priority NA:", int(na.sum()), "| Impact NS:", int(ns.sum()), "| both:", int((na & ns).sum()),
      "| Urgency recorded on those:", int((na & df["Urgency"].str.strip().ne("")).sum()))

Table 8b. Tickets per Impact x Urgency pair (rows) and Priority (columns)


Priority             1    2     3      4      5    NA
Impact Urgency                                       
1      1             3    0     0      0      0     0
2      2             0  689     0      0      0     0
       3             0    3     0      0      0     0
3      1             0    2     0      0      0     0
       2             0    3     0      0      0     0
       3             0    0  5077      0      0     0
       4             0    0   144      0      0     0
       5             0    0     0      8      0     0
4      2             0    0     3      0      0     0
       3             0    0    97      0      0     0
       4             0    0     0  22170      0     0
       5             0    0     0    286      0     0
5      1             0    0     1      0      0     0
       2             0    0     1      0      0     0
       3             0    0     0     11      0     0
       4             0    0     0    242      0     0
       5             0    0     0      0  16485     0
       5 - Very Low  0    0     0      0      1     0
NS     3             0    0     0      0      0  1348
       4             0    0     0      0      0    32

Impact x Urgency pairs that map to more than one priority: 0 of 20
Priority NA: 1380 | Impact NS: 1380 | both: 1380 | Urgency recorded on those: 1380


<details>
<summary><b>Code notes, line by line</b></summary>

- `pm = pd.crosstab([df["Impact"], df["Urgency"]], df["Priority"])` : Counts tickets for every Impact x Urgency pair (rows) and Priority (columns).
- `display(pm)` : Shows the matrix: each row should have only one non-zero column if Priority is a lookup.
- `((pm > 0).sum(axis=1) > 1).sum()` : Number of Impact x Urgency pairs that occur with more than one Priority; 0 proves the lookup.
- `na, ns = ...eq("NA"), ...eq("NS")` : Flags tickets with Priority 'NA' and Impact 'NS'.
- `(na & ns).sum() / Urgency recorded on those` : Checks the two placeholders fall on the same tickets and that Urgency is still filled in on them.

</details>

**Result.** Table 8a shows the placeholders `NS` (Impact) and `NA` (Priority) on 1,380 tickets each, one `5 - Very Low` Urgency label, Status Closed on 46,597 of 46,606 tickets, only 3 P1 and 697 P2 tickets, and a single Alert_Status value. In Table 8b every Impact x Urgency row has exactly one non-zero Priority column (0 of 20 pairs map to more than one priority): Priority is a lookup of Impact and Urgency.

## 9. Handle_Time_hrs format

In [12]:
print("Table 9. Handle_Time_hrs values containing a comma:", df["Handle_Time_hrs"].str.contains(",").sum())

Table 9. Handle_Time_hrs values containing a comma: 46349


<details>
<summary><b>Code notes, line by line</b></summary>

- `df["Handle_Time_hrs"].str.contains(",").sum()` : Counts values containing a comma, a sign of a decimal comma or digit grouping that pandas cannot convert directly.

</details>

**Result.** 46,349 of 46,606 values contain a comma (Table 9), so they cannot be converted to numbers directly; `03_cleaning`, section 8, shows what they are.

## 10. Date formats

In [13]:
print("Table 10. Date format checks")
parts = df["Open_Time"].str.extract(r"^(\d{1,2})-(\d{1,2})-(\d{4})").apply(pd.to_numeric, errors="coerce")
print("Open_Time rows not matching dd-mm-yyyy:", parts[0].isna().sum())
print("First part max:", parts[0].max(), "| Second part max:", parts[1].max(), "| Years:", parts[2].min(), "to", parts[2].max())

for c in ["Resolved_Time", "Close_Time", "Reopen_Time"]:
    s = df[c].str.strip()
    s = s[s != ""]
    print(c, "| non-blank:", len(s), "| matching dd-mm-yyyy hh:mm:", s.str.match(r"^\d{1,2}-\d{1,2}-\d{4} \d{1,2}:\d{2}").sum())

Table 10. Date format checks
Open_Time rows not matching dd-mm-yyyy: 0
First part max: 31 | Second part max: 12 | Years: 2012 to 2014
Resolved_Time | non-blank: 44826 | matching dd-mm-yyyy hh:mm: 44826
Close_Time | non-blank: 46606 | matching dd-mm-yyyy hh:mm: 46606
Reopen_Time | non-blank: 2284 | matching dd-mm-yyyy hh:mm: 2284


<details>
<summary><b>Code notes, line by line</b></summary>

- `df["Open_Time"].str.extract(r"^(\d{1,2})-(\d{1,2})-(\d{4})")` : Splits the date part into three numbers with a regular expression.
- `.apply(pd.to_numeric, errors="coerce")` : Converts them to numbers; anything that does not match becomes NaN.
- `parts[0].isna().sum()` : Rows that do not match the pattern.
- `parts[0].max(), parts[1].max()` : If the first part reaches 31 and the second only 12, the format is day-month-year, not month-day-year.
- `for c in ["Resolved_Time", ...]: s.str.match(...)` : For the other date columns, counts non-blank values matching dd-mm-yyyy hh:mm.

</details>

**Result.** Every non-blank date matches dd-mm-yyyy hh:mm (Table 10). The first part reaches 31 and the second only 12, so the order is day-month-year; Open_Time years run from 2012 to 2014.

## 11. Findings
1. **Storage.** All 25 columns are text (Table 3a) and `isna()` finds no nulls (Table 3b); missing values are blanks or placeholders (Table 4: `NA` in Priority and `NS` in Impact on 1,380 tickets each). Dates are all dd-mm-yyyy hh:mm, 2012 to 2014 (Table 10).
2. **Integrity.** Incident_ID is unique and never blank, and there are no duplicate rows (Table 6). The database itself enforces none of this (Table 1c: no constraints).
3. **Columns to drop.** number_cnt holds uniform random values between 0 and 1, all unique (Table 7), and is not in the project brief; Alert_Status has one value (Table 8a). Status is Closed on 46,597 of 46,606 tickets (Table 8a) and is only known after handling.
4. **Priority is a lookup of Impact and Urgency.** No Impact x Urgency pair maps to more than one priority (Table 8b: 0 of 20). Priority NA and Impact NS fall on the same 1,380 tickets, all with an Urgency recorded. So Impact and Urgency are excluded from the Track 1 headline features (a model given them only relearns the matrix), and the 1,380 unlabelled tickets are excluded from Track 1.
5. **Severe imbalance.** 3 tickets are P1 and 697 are P2, about 1.5% together (Table 8a), so P1 and P2 are merged into "High" for Track 1.
6. **Label fix.** Urgency has one `5 - Very Low` (Table 8a), recoded to 5.
7. **Track 4 target.** Only 560 tickets have a related change (46,046 of 46,606 blank, Table 4), and Category has a single "request for change" (Table 8a), so the RFC target is Has_Related_Change.
8. **Handle_Time_hrs.** 46,349 values contain commas (Table 9); the conversion is checked in `03_cleaning`.